Going to try with matplot lib

I used chatgpt to help me generate some of this initial code

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors

Read in dataframes, and then make a list of dataframes so I can run through them all

In [ ]:
#election years
# 
df_2005 = pd.read_csv('DC-migration-2005.csv')
df_2009 = pd.read_csv('DC-migration-2009.csv')
df_2013 = pd.read_csv('DC-migration-2013.csv')
df_2017 = pd.read_csv('DC-migration-2017.csv')
df_2021 = pd.read_csv('DC-migration-2021.csv')


#non election years

df_2015 = pd.read_csv('DC-migration-2015.csv')
df_2016 = pd.read_csv('DC-migration-2016.csv')
df_2018 = pd.read_csv('DC-migration-2018.csv')
df_2019 = pd.read_csv('DC-migration-2019.csv')



#make a list

dfs = [df_2005, df_2009, df_2013, df_2015, df_2016, df_2017, df_2018, df_2019, df_2021]

Drop the initial columns that were carried over from the indexes and messing up this data.

In [ ]:
df_2005 = df_2005.drop(columns=["Unnamed: 0"])
df_2009 = df_2009.drop(columns=["Unnamed: 0"])
df_2013 = df_2013.drop(columns=["Unnamed: 0"])
df_2015 = df_2015.drop(columns=["Unnamed: 0"])
df_2016 = df_2016.drop(columns=["Unnamed: 0"])
df_2017 = df_2017.drop(columns=["Unnamed: 0"])
df_2018 = df_2018.drop(columns=["Unnamed: 0"])
df_2019 = df_2019.drop(columns=["Unnamed: 0"])
df_2021 = df_2021.drop(columns=["Unnamed: 0"])


In [ ]:
df_2005.tail()

Add in a state year column so in order to be able to loop through the dataframes


In [ ]:
df_2005['year'] = 2005
df_2009['year'] = 2009
df_2013['year'] = 2013
df_2015['year'] = 2015
df_2016['year'] = 2016
df_2017['year'] = 2017
df_2018['year'] = 2018
df_2019['year'] = 2019
df_2021['year'] = 2021


Ok, multiplying by 100,000 to make the migrated_adjusted and moe_adjusted more readable 

In [ ]:
for df in dfs:
    df['migrated_adjusted'] = df['migrated_adjusted'] * 100000
    df['moe_adjusted'] = df['moe_adjusted']* 100000

In the dataframes for non-election years, need to populate winner_percent_of_votes and winning_party columns with info from the most recent election year

In [ ]:
df_2015['winning_party'] = df_2017['winning_party']
df_2016['winning_party'] = df_2017['winning_party']

df_2018['winning_party'] = df_2021['winning_party']
df_2019['winning_party'] = df_2021['winning_party']


df_2015['winner_percent_of_votes'] = df_2017['winner_percent_of_votes']
df_2016['winner_percent_of_votes'] = df_2017['winner_percent_of_votes']

df_2018['winner_percent_of_votes'] = df_2021['winner_percent_of_votes']
df_2019['winner_percent_of_votes'] = df_2021['winner_percent_of_votes']

#reset the list -- for some reason it needed me to run this otherwise it has like two different versions of these dfs saved
dfs = [df_2005, df_2009, df_2013, df_2015, df_2016, df_2017, df_2018, df_2019, df_2021]

df_2015.head()

Make sure all the values I need are numeric.

In [ ]:
for df in dfs:

    df["migrated_adjusted"] = pd.to_numeric(df["migrated_adjusted"])
    df["moe_adjusted"] = pd.to_numeric(df["moe_adjusted"])
    df["winner_percent_of_votes"] = pd.to_numeric(df["winner_percent_of_votes"])

In [ ]:
import matplotlib.pyplot as plt

for df in dfs:

    # Keep only the 8 states with the largest migration values
    df = df.nlargest(18, "migrated_adjusted")
    df.reset_index(drop=True, inplace=True)

    party_colors = {
    "REPUBLICAN": "firebrick",
    "DEMOCRAT": "royalblue"
    }

    colors = [party_colors.get(party, "gray") for party in df["winning_party"]]

    plt.figure(figsize=(10, 5))

    plt.barh(
        df["state"],                   # Labels on the x-axis
        df["migrated_adjusted"],       # Height of each bar
        # xerr=df["moe_adjusted"],       # Error bars (+/-)
        # capsize=3                      # Small horizontal caps on error bars
        color=colors

    )

    # Add labels and a title
    plt.title(f'Migration to DC in {df.loc[0, 'year']}')
    plt.xlabel("Moved to DC (adjusted for population)")
    plt.ylabel("State")

    plt.xticks(rotation=90) # Rotate state names so they don't overlap

    plt.tight_layout()  # Automatically adjust spacing

    plt.show() # Display the chart


Saving top 20 and top 15 and top 10 for each of the election years so I can refer back

In [ ]:
top_dfs = {}

top_15_dfs = []

for df in dfs:
    year = df["year"].iloc[0]  # Get the year from the first row

    # top_dfs[f"{year}_top10"] = df.nlargest(10, "migrated_adjusted").reset_index(drop=True)
    # top_dfs[f"{year}_top15"] = df.nlargest(15, "migrated_adjusted").reset_index(drop=True)
    # top_dfs[f"{year}_top20"] = df.nlargest(20, "migrated_adjusted").reset_index(drop=True)

    top_15_dfs.append(df.nlargest(15, "migrated_adjusted").reset_index(drop=True))

# top_dfs["2021_top20"] #testing

Going to try some maps! Want to make contrasting 2021 and 2017 maps

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px

from PIL import Image
from matplotlib.patches import Patch, Circle

Load US states geometries

In [ ]:
# states = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip")

Merge data and geometry

In [ ]:
# merged = states.merge(top_dfs["2021_top20"], left_on="NAME", right_on="state", how="left")

Plot

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# merged.plot(
#     column="migrated_adjusted",          # or any numeric/boolean column
#     cmap="coolwarm",         # color scheme
#     linewidth=0.8,
#     edgecolor="black",
#     legend=True,
#     ax=ax
# )

# ax.set_title("Migration to DC in 2021")
# ax.axis("off")

# plt.show()

Trying plotly instead of the above:



cleaned_for_plotly:

Make  a new column that adjusts the sign of winner_percent_of_votes so I can map with colors both corresponding to each party, but also how red or blue (how high the percent that voted for that presidential candidate.

If DEMOCRAT  -> keep value positive
If REPUBLICAN -> make value negative

In [ ]:
for i in range(len(top_15_dfs)):
    top_15_dfs[i]["percent_won_and_party"] = np.where(
        top_15_dfs[i]["winning_party"] == "DEMOCRAT",
        top_15_dfs[i]["winner_percent_of_votes"],    # keep as-is (positive)
        top_15_dfs[i]["winner_percent_of_votes"]     # flip sign to negative
    )

In [ ]:
top_15_dfs[i].head()

Plotly also requires 2 letter codes for states, so need to fix that

In [ ]:
us_state_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC"
}

for i in range(len(top_15_dfs)):
    top_15_dfs[i]["state_code"] = top_15_dfs[i]["state"].map(us_state_abbrev)

Map for 2016

remember that dfs = [df_2005, df_2009, df_2013, df_2015, df_2016, df_2017, df_2018, df_2019, df_2021]

In [ ]:

fig_2016 = px.choropleth(
    top_15_dfs[4],
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party", 
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2016.show()

Map for 2017:

In [ ]:

fig_2017 = px.choropleth(
    top_15_dfs[5],
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2017.show()



Map for 2018

In [ ]:
fig_2018 = px.choropleth(
    top_15_dfs[6],
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2018.show()

Map for 2019

In [ ]:
fig_2019 = px.choropleth(
    top_15_dfs[7],
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2019.show()

Map for 2021

In [ ]:
top_15_dfs[8].head()


In [ ]:
fig_2021 = px.choropleth(
    top_15_dfs[8],
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2021.show()



In [ ]:
top_15_dfs[8]

In [ ]:
top_15_dfs[4].head(10)


In [ ]:
fig_2009 = px.choropleth(
    top_15_dfs[1],
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2009.show()

Contrasting with non election years

In [ ]:
fig_2009 = px.choropleth(
    top_15_dfs[1],
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2009.show()

Save them as .svgs

In [ ]:
# 4. Export the map to a vector SVG file


fig_2009.write_image("2009_map.svg", engine="kaleido")
fig_2016.write_image("2016_map.svg", engine="kaleido")
fig_2017.write_image("2017_map.svg", engine="kaleido")
fig_2019.write_image("2019_map.svg", engine="kaleido")
fig_2021.write_image("2021_map.svg", engine="kaleido")

print("Map successfully exported as 'choropleth_map.subplots.svg'!")

KeyboardInterrupt: 

Error in run_read_loop.
Traceback (most recent call last):
  File "/Users/abbymihaly/.venvs/lede/lib/python3.13/site-packages/choreographer/_brokers/_async.py", line 230, in read_loop
    raise protocol.DevtoolsProtocolError(response)
choreographer.protocol.DevtoolsProtocolError: {'id': 0, 'error': {'code': -32001, 'message': 'Session with given id not found.'}}
Exception in callback Broker.run_read_loop.<locals>.check_read_loop_error() at /Users/abbymihaly/.venvs/lede/lib/python3.13/site-packages/choreographer/_brokers/_async.py:126
handle: <Handle Broker.run_read_loop.<locals>.check_read_loop_error() at /Users/abbymihaly/.venvs/lede/lib/python3.13/site-packages/choreographer/_brokers/_async.py:126>
Traceback (most recent call last):
  File "/Users/abbymihaly/.local/share/uv/python/cpython-3.13.13-macos-aarch64-none/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ab

Generating top 15 lists!

In [ ]:
with open("top15.txt", "w") as f:
    for i in range(len(top_15_dfs)):
        print(top_15_dfs[i]['year'][0], file=f)
        states = []
        for state in top_15_dfs[i]['state']:
            states.append(state)
        print(*states, sep="\n", file=f)
        print("---", file=f)